# OncoVision: train the 3D lung nodule model (3D U-Net on LIDC-IDRI chest CT)

**How to run (important, so nothing gets lost):**
1. Right panel: **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.
2. Top right: **Save Version → Save & Run All (Commit) → Save**. Do *not* use the normal Run All.
3. Close the tab if you like. It runs in the background for about 2-3 hours.
4. Later open the notebook page → **Output** tab → download `oncovision_lung3d.zip` and send the printed scores to Claude.

Data: LIDC-IDRI chest CTs with nodule outlines (641 patients), from a public Hugging Face NIfTI conversion (totalol/lungct), no login needed. Original data: The Cancer Imaging Archive, CC BY 3.0.


In [ ]:
!git clone -q -b vaishnavi/optimistic-noether-034odi https://github.com/Vaishnavi220506/Medical.git /kaggle/working/oncovision
%cd /kaggle/working/oncovision
!pip install -q -r requirements.txt huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings!")


## 1 + 2. Download, prepare and train (about 2-3 hours)

Each CT is downloaded, converted (lung window, 1 mm grid, cropped to the lungs) and deleted again, so the 38 GB of raw data never fill the disk. Then the model trains. 100 patients are held out: 40 for validation, 60 for a final test score.


In [ ]:
!python -m oncovision.training.train_lung3d --hf --prepared-dir /tmp/lung3d --epochs 60 --samples-per-epoch 800 \
    --patch 96 --batch-size 4 --val-cases 40 --test-cases 60 --val-every 5 --workers 4 --out weights/lung_ct/segmenter3d.pt


## 3. Scores and package

In [ ]:
import torch, json
meta = torch.load("weights/lung_ct/segmenter3d.pt", map_location="cpu", weights_only=True)["meta"]
m = meta["metrics"]
for split in ("validation", "test"):
    if split in m:
        print(split.upper(), {k: v for k, v in m[split].items() if k != "by_threshold"})
print("SCORES (send these to Claude):", json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "by_threshold"} for k, v in m.items()}), "best epoch", meta["epoch"])
!python scripts/export_weights.py --src weights/lung_ct --dst export_weights/lung_ct
!cd /kaggle/working && zip -qj oncovision_lung3d.zip oncovision/export_weights/lung_ct/segmenter3d.pt && ls -la oncovision_lung3d.zip
